# Project evaluation: Human-Centered Evaluation of Your Collaborative Agent

## **Important! Make a copy of this file...**
- [ ] This is a Colab template that you should not modify.
- [ ] Please make your own copy, by selecting e.g. `File > Save a copy in Drive`.
- [ ] Afterwards, make your file identifiable by renaming it to be `AIAgents-S26-youragenthere.ipynb`.


## Overview

In this assignment, you will evaluate your team project agent using two complementary methods:

- Intrinsic Evaluation — an automated evaluation (e.g., LLM-as-a-judge or code-based metric).
- Extrinsic Evaluation — a small-scale user study with real participants outside the class.

You will design, implement, and analyze these evaluations, then summarize your work in a structured write-up that includes both your results and reflections.

You will also propose one additional evaluation type you could run in the future (but won’t conduct for this assignment).

## Learning Objectives:
By the end of this assignment, you should be able to:

- Apply intrinsic metrics for automated performance measurement of conversational agents.
- Design and conduct a small-scale user study to capture user experience and task outcomes.
- Use the SPHERE Evaluation Card to communicate your evaluation design clearly.
- Draw evidence-based conclusions about an agent’s performance.

## Grading Rubric (Total: 100 points)

| Category | Points | Description |
|----------|--------|-------------|
| **Intrinsic Evaluation Design & Implementation** | 30 | 1–3 well-defined metrics tied to agent goals; correct LLM/code implementation; tested on ≥3 diverse examples with meaningful results. |
| **Extrinsic Evaluation Design & Execution** | 30 | Clear, realistic protocol; relevant participants/tasks; both quantitative and qualitative data collected and documented. |
| **SPHERE Evaluation Cards** | 5 | Two completed JSON cards (intrinsic & extrinsic); accurate, detailed, and aligned with evaluation methods. |
| **Results & Conclusions** | 20 | Clear presentation of results; at least one supported conclusion per evaluation with convincing evidence. |
| **Proposed Additional Evaluation** | 10 | Thoughtful and relevant; clear design outline and strong justification for future use. |
| **Clarity & Effort** | 5 | Well-organized, clearly written submission demonstrating strong effort and attention to detail. |




## Step 1: Setup

(You'll need some kind of setup code in your repo, here is an example)

Here, we will do a similar round of setup as in Assignment 1, installing necessary packages for modeling calling. It also copies over the interactive interface I hacked in A1.
**If there are code you would like to reuse, please feel free to copy over here.**

In [ ]:
# same setup as in assignment 1
# this installs a specific earlier version of the library to get the hack
# for the GradioUI chat interface
!pip install 'smolagents[litellm] @ git+https://github.com/huggingface/smolagents.git@df2757b'
!pip install 'smolagents[gradio] @ git+https://github.com/huggingface/smolagents.git@df2757b'

!pip install 'boto3'
!pip install 'markdownify'

  Cloning https://github.com/huggingface/smolagents.git (to revision df2757b) to /tmp/pip-install-5rpalh93/smolagents_d65dc64b28d74b0986ed42307bb356d3
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/smolagents.git /tmp/pip-install-5rpalh93/smolagents_d65dc64b28d74b0986ed42307bb356d3
  Running command git checkout -q df2757b
  Resolved https://github.com/huggingface/smolagents.git to commit df2757b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.1 MB/s eta 0:00:00
  Created wheel for smolagents: filename=smolagents-1.22.0.dev0-py3-none-any.whl size=141734 sha256=43ac2a7a392dbe8f9d071e6d423e0ac6d6296e8ecd959591954766fe0428942c
  Stored in directory: /tmp/pip-ephem-wheel-cache-5nbbsf1h/wheels/03/e4/7b/da95c516518b730dbbaa66f5e5

In [ ]:
# same interface as A1

from smolagents.gradio_ui import GradioUI, pull_messages_from_step

import os
import re
import shutil
from pathlib import Path
from typing import Generator

from smolagents.agent_types import AgentAudio, AgentImage, AgentText
from smolagents.agents import MultiStepAgent, PlanningStep
from smolagents.memory import ActionStep, FinalAnswerStep, TaskStep
from smolagents.models import ChatMessageStreamDelta, MessageRole, agglomerate_stream_deltas
from smolagents.utils import _is_package_available


class GradioUIChat(GradioUI):
  def create_app(self):
    import gradio as gr
    with gr.Blocks(theme="ocean", fill_height=True) as demo:
      # Add session state to store session-specific data
      session_state = gr.State({})
      stored_messages = gr.State([])
      file_uploads_log = gr.State([])


      with gr.Sidebar():
        gr.Markdown(
          f"# {self.name.replace('_', ' ').capitalize()}"
          "\n> This web ui allows you to interact with a `smolagents` agent that can use tools and execute steps to complete tasks."
          + (f"\n\n**Agent description:**\n{self.description}" if self.description else "")
        )
        # If an upload folder is provided, enable the upload feature
        if self.file_upload_folder is not None:
          upload_file = gr.File(label="Upload a file")
          upload_status = gr.Textbox(label="Upload Status", interactive=False, visible=False)
          upload_file.change(
            self.upload_file,
            [upload_file, file_uploads_log],
            [upload_status, file_uploads_log],
          )

          gr.HTML(
            "<br><br><h4><center>Powered by <a target='_blank' href='https://github.com/huggingface/smolagents'><b>smolagents</b></a></center></h4>"
          )


      # Main chat interface
      chatbot = gr.Chatbot(
        label="Agent",
        type="messages",
        avatar_images=(
          None,
          "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/smolagents/mascot_smol.png",
        ),
        resizeable=True,
        scale=1,
        latex_delimiters=[
          {"left": r"$$", "right": r"$$", "display": True},
          {"left": r"$", "right": r"$", "display": False},
          {"left": r"\[", "right": r"\]", "display": True},
          {"left": r"\(", "right": r"\)", "display": False},
        ],
      )
      with gr.Group():
        text_input = gr.Textbox(
            lines=2,
            label="Chat Message",
            container=False,
            placeholder="Enter your prompt here and press Shift+Enter or press the button",
        )
        submit_btn = gr.Button("Submit")

      text_input.submit(
        self.log_user_message,
        [text_input, file_uploads_log],
        [stored_messages, text_input, submit_btn],
      ).then(
          self.interact_with_agent, [stored_messages, chatbot, session_state], [chatbot]
      ).then(
        lambda: (
          gr.Textbox(interactive=True, placeholder="Enter your prompt here and press Shift+Enter or the button"),
          gr.Button(interactive=True),
        ),
        None,
        [text_input, submit_btn],
      )

      submit_btn.click(
        self.log_user_message,
        [text_input, file_uploads_log],
        [stored_messages, text_input, submit_btn],
      ).then(
          self.interact_with_agent, [stored_messages, chatbot, session_state], [chatbot]
      ).then(
        lambda: (
          gr.Textbox(interactive=True, placeholder="Enter your prompt here and press Shift+Enter or the button"),
          gr.Button(interactive=True),
        ),
        None,
        [text_input, submit_btn],
      )
      chatbot.clear(self.agent.memory.reset)
      return demo

  def stream_to_gradio(
      agent,
      task: str,
      task_images: list | None = None,
      reset_agent_memory: bool = False,
      additional_args: dict | None = None,
  ) -> Generator:
    """Runs an agent with the given task and streams the messages from the agent as gradio ChatMessages."""
    if not _is_package_available("gradio"):
      raise ModuleNotFoundError(
          "Please install 'gradio' extra to use the GradioUI: `pip install 'smolagents[gradio]'`"
          )
    accumulated_events: list[ChatMessageStreamDelta] = []
    if reset_agent_memory:
      agent.memory.reset()
      agent.monitor.reset()
    final_answer = None
    def stream_event(event):
      if isinstance(event, ActionStep | PlanningStep | FinalAnswerStep):
        for message in pull_messages_from_step(
            event,
            # If we're streaming model outputs, no need to display them twice
            skip_model_outputs=getattr(agent, "stream_outputs", False),
        ):
          yield message
        accumulated_events = []
      elif isinstance(event, ChatMessageStreamDelta):
        accumulated_events.append(event)
        text = agglomerate_stream_deltas(accumulated_events).render_as_markdown()
        yield text

      # run the agent only until there's a user input
      step_number = 1
      agent.memory.steps.append(TaskStep(task=task, task_images=[]))
      stream_event(agent.memory.steps[-1])
      while final_answer is None and step_number <= agent.max_steps:
        step = ActionStep(step_number=step_number)
        # Run one step
        final_answer = agent.step(step)
        agent.memory.steps.append(step)
        step_number += 1
        stream_event(agent.memory.steps[-1])
        print(str([tc.dict() for tc in step.tool_calls]))
        if isinstance(step, ActionStep) and \
          step.tool_calls and \
          ("user_input_tool" in str([tc.dict() for tc in step.tool_calls]) or "user_input_tool" in step.code_action):
          break



## Step 2: Load the agent to test

(This is just an example, you will probably want to write this in your project repo as its own script, just document what scripts to run and commit outputs from them and link the files here/in your project Readme)

Here, you will set up your AWS and Huggingface Access Keys to reload the agent you created in Assignment 1. This will be the agent you test.

In [ ]:
import os
# set the keys to be in the env

from google.colab import userdata
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_REGION_NAME"] = "us-east-1"

In [ ]:
import os
from huggingface_hub import login
from smolagents import CodeAgent

# If you forgot how to get the tokens, check back at A1
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
login(token=os.getenv("HF_TOKEN"))

# Use the agent you created and uploaded in A1.
AGENT_PATH ="fill this in here i.e. username/travel_agent"
collab_agent = CodeAgent.from_hub(AGENT_PATH, trust_remote_code=True)
# make sure you succeeded
collab_agent.run("Hello there!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

╭────────────────────────────────────────── New run - travel_assistant ───────────────────────────────────────────╮
│                                                                                                                 │
│ Hello there!                                                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.deepseek.r1-v1:0 ────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                   
                                                                                                                   
Thought: The user greeted me, so I should respond politely and ask how I can assist them. Since this is the first  
message, there's no conversation history to analyze yet. I'll use the `user_input_tool` to send a friendly response
while maintaining an open question format.                                                                         
                                                                                                                   
<code>                                                                                                             
response = user_input_tool(                                                                                        
    context="Welcome message to start conversation",                                                               
    question="Hello! How can I assist you today?"                                                                  
)                                                                                                                  
final_answer(response)                                                                                             
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  response = user_input_tool(                                                                                      
      context="Welcome message to start conversation",                                                             
      question="Hello! How can I assist you today?"                                                                
  )                                                                                                                
  final_answer(response)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Welcome message to start conversation 
 Hello! How can I assist you today?

[Step 1: Duration 2.24 seconds| Input tokens: 2,452 | Output tokens: 305]

'Welcome message to start conversation \n Hello! How can I assist you today?'

In [ ]:
# You can also try the interface interaction

collab_agent.memory.reset()
collab_agent.monitor.reset()
GradioUIChat(collab_agent).launch(False)

## Step 3: Intrinsic evaluation

- Define **1–3 metrics** that capture an important aspect of your agent’s performance (e.g., factual accuracy, task completeness, response brevity, trade-off clarity).
- Implement these metrics using either:
  - **LLM-as-a-judge** (provide prompt & scoring rubric to an LLM, return numerical score)
  - **Code-based scoring** (string matching, word count, time to completion, etc.)
- Test the metrics on **at least 3 example interactions** with your agent
  - This one will require you to interact with your agent for at least three times, ideally with you simulating diverse personas to create diverse use cases -- Think of how they would represent different use scenarios or edge cases!
  - If you feel motivated, you could also try creating simulated LLM personas that would have simulated conversation with the agent, similar to [Co-Gym](https://arxiv.org/abs/2412.15701).
- Save both the metric definitions and results.
- To submit this part, I recommend you just link give a link to a branch on your project's github repository.


Below is some example code implementing the same verifier using LLM-as-judge and rule-based python function (though the rule-based method would miss edge cases for sure). The default starter implementation provided below uses the [Judges](https://github.com/quotient-ai/judges) library, but you can also create it from scratch or use another library. Some useful reference links, from the Huggingface Cookbook:
- [Using LLM-as-a-judge 🧑‍⚖️ for an automated and versatile evaluation](https://huggingface.co/learn/cookbook/en/llm_judge) -- A general walkthrough on LLM-as-judge (check it out / see also the documentation)
- [Evaluating AI Search Engines with judges - the open-source library for LLM-as-a-judge evaluators ⚖️](https://huggingface.co/learn/cookbook/en/llm_judge_evaluating_ai_search_engines_with_judges_library) -- More examples on the `Judges` library.

### Intrinsic evaluation — SAGE

Two metrics scoring SAGE against its own pedagogical contract (CLAUDE.md). Implementation lives under `evaluation/` in this repo.

**Branch link:** https://github.com/cyrilagbewalikoku-oss/G10COS498_AiUseTutor/tree/AgentV2.1

#### Metric 1 — Front-Loading Discipline (rule-based)

Two sub-checks per SAGE turn:

- **Question Discipline** — count of `?` characters must be ≤ 1. Catches stacked questions like *"What did you mean? And why?"*.
- **Pre-Pause Length** — sentences before the first `?` (or whole turn if no `?`) must be ≤ 4. Catches walls of text before SAGE invites the learner in.

Source: `evaluation/metrics/front_loading.py`.

#### Metric 2 — Answer-First Adherence (LLM-as-judge, two-stage)

Applied only to SAGE turns whose immediately preceding learner turn is a direct question.

- **Stage 1 — Classifier (Haiku 4.5):** labels the learner message as `yes_no`, `factual`, `open`, or `not_a_question`. Only `yes_no` and `factual` qualify for grading.
- **Stage 2 — Grader (Sonnet 4.6):** labels SAGE's reply as `answered_first`, `answered_and_followed_up`, `redirected_without_answer`, or `non_answer`.
- **Pass condition:** `behavior in {answered_first, answered_and_followed_up}`.

Source: `evaluation/metrics/answer_first.py`.

#### Three transcript sources

The runner combines:

1. **Authored** — 7 hand-written transcripts in `examples/interactions/{positive,negative}/*.md`.
2. **Simulated** — 3 persona-driven sessions (`novice-curious`, `skeptical-engineer`, `fatigued-returner`) generated by `evaluation.personas.simulator` driving Claude Opus 4.7 (SAGE) against Claude Haiku 4.5 (persona learner).
3. **Exported** — chats downloaded from the SAGE Streamlit app's "Export chat" panel, dropped into `evaluation/fixtures/exports/`.

#### Note on the starter `judges` library

The notebook starter recommends the `judges` library. We chose to roll our own thin LLM-as-judge harness over the Anthropic SDK (`evaluation/metrics/judge.py`) because (a) SAGE itself is built on the Anthropic SDK so we keep one stack and one API key, (b) we needed two-stage gating (classifier → grader) which is cleaner with direct calls, and (c) the JSON-keyed cache means re-runs against unchanged transcripts cost zero tokens. The cells below show the actual code.


In [ ]:
# Setup — installs the project (which pulls anthropic) and dev extras (pytest).
# Run from the repo root.
!pip install -e ".[dev]"

# Optional: set your Anthropic API key for Metric 2. Skip with `--no-judge` to run rule-based only.
# import os; os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."


In [ ]:
# Metric 1: Front-Loading Discipline (rule-based)
# Source: evaluation/metrics/front_loading.py

import re
from evaluation.metrics.transcript import Transcript

_SENTENCE_END_RE = re.compile(r"[.!]")
_PRE_PAUSE_THRESHOLD = 4


def _count_sentences(text: str) -> int:
    return len(_SENTENCE_END_RE.findall(text))


def _score_turn(text: str) -> dict:
    qmarks = text.count("?")
    qd_passed = qmarks <= 1
    pre_pause_text = text[: text.index("?")] if "?" in text else text
    sentences = _count_sentences(pre_pause_text)
    pp_passed = sentences <= _PRE_PAUSE_THRESHOLD
    return {
        "question_discipline": {"question_marks": qmarks, "passed": qd_passed},
        "pre_pause": {"sentences": sentences, "passed": pp_passed},
        "passed": qd_passed and pp_passed,
    }


def score_front_loading(transcript: Transcript) -> list[dict]:
    """Score every SAGE turn. Learner turns are skipped."""
    return [
        {**_score_turn(t.text), "turn_index": t.index}
        for t in transcript.turns if t.speaker == "sage"
    ]


# Quick demo on illustrative SAGE turns:
demo_turns = [
    "Welcome to SAGE.",
    "Nice work. What stood out to you in that response?",
    "What did you mean? And why did that surprise you?",
    "First sentence. Second sentence. Third sentence. Fourth sentence. Fifth sentence. What do you think?",
]
for t in demo_turns:
    r = _score_turn(t)
    verdict = "PASS" if r["passed"] else "FAIL"
    print(f"[{verdict}] q-marks={r['question_discipline']['question_marks']} "
          f"pre-pause-sentences={r['pre_pause']['sentences']}: {t[:60]}")


[PASS] q-marks=0 pre-pause-sentences=1: Welcome to SAGE.
[PASS] q-marks=1 pre-pause-sentences=1: Nice work. What stood out to you in that response?
[FAIL] q-marks=2 pre-pause-sentences=0: What did you mean? And why did that surprise you?
[FAIL] q-marks=1 pre-pause-sentences=5: First sentence. Second sentence. Third sentence. Fou


In [ ]:
# Metric 2: Answer-First Adherence (two-stage LLM-as-judge)
# Source: evaluation/metrics/answer_first.py

from evaluation.metrics.answer_first import (
    AnthropicJudge,
    score_answer_first,
    CLASSIFIER_SYSTEM,
    GRADER_SYSTEM,
    QUALIFYING_KINDS,
    PASSING_BEHAVIORS,
)

print("=== Stage 1: classifier prompt (Haiku 4.5) ===")
print(CLASSIFIER_SYSTEM)
print()
print("=== Stage 2: grader prompt (Sonnet 4.6) ===")
print(GRADER_SYSTEM)
print()
print(f"QUALIFYING_KINDS  = {QUALIFYING_KINDS}")
print(f"PASSING_BEHAVIORS = {PASSING_BEHAVIORS}")


=== Stage 1: classifier prompt (Haiku 4.5) ===
You classify whether a learner's message to a tutor is a "direct question"
that requires a substantive answer before any follow-up coaching question.

Categories:
- "yes_no": a question whose natural answer is yes/no
            (e.g., "Is it okay to paste customer data into ChatGPT?")
- "factual": a question asking for a definition or explanation
            (e.g., "What is a hallucination?", "How does prompt caching work?")
- "open":   an open-ended invitation (e.g., "Tell me about your experience...")
- "not_a_question": statement, reaction, or compliance message.

Return ONLY a JSON object with keys "kind" and "reasoning". No prose outside JSON.

=== Stage 2: grader prompt (Sonnet 4.6) ===
You grade whether a tutor answered a learner's direct question before redirecting.

Categories for the tutor's reply:
- "answered_first":           tutor gave a substantive answer; may have stopped there.
- "answered_and_followed_up": tutor answered 

In [ ]:
# End-to-end: load all 3 transcript sources, run both metrics, save dated results.
# Equivalent to: python -m evaluation.run_evaluation
from evaluation.run_evaluation import (
    _load_authored, _load_simulated, _load_exported,
    _aggregate, _git_head, JUDGE_CACHE_PATH, RESULTS_DIR, TranscriptScored,
)
from evaluation.metrics.front_loading import score_front_loading
from evaluation.metrics.answer_first import AnthropicJudge, score_answer_first
from datetime import datetime, timezone
import json, os

transcripts = _load_authored() + _load_simulated() + _load_exported()
print(f"Loaded {len(transcripts)} transcripts: "
      f"{sum(1 for t in transcripts if t.origin=='authored')} authored, "
      f"{sum(1 for t in transcripts if t.origin=='simulated')} simulated, "
      f"{sum(1 for t in transcripts if t.origin=='exported')} exported")

# Set ANTHROPIC_API_KEY before running this cell to enable Metric 2.
judge = AnthropicJudge(cache_path=JUDGE_CACHE_PATH) if "ANTHROPIC_API_KEY" in os.environ else None
print(f"Judge: {'AnthropicJudge (Metric 2 enabled)' if judge else 'None (Metric 2 skipped)'}")

scored = []
for t in transcripts:
    fl = score_front_loading(t)
    af = score_answer_first(t, judge=judge) if judge else []
    scored.append(TranscriptScored(transcript=t, front_loading=fl, answer_first=af))

aggregates = _aggregate(scored)
print()
print("Overall pass rates:")
for metric in ("front_loading", "answer_first"):
    m = aggregates["overall"][metric]
    print(f"  {metric:15s} {m['passed']}/{m['applicable']} ({m['rate']})")


Loaded 10 transcripts: 7 authored, 3 simulated, 0 exported
Judge: AnthropicJudge (Metric 2 enabled)

Overall pass rates:
  front_loading   23/53 (0.434)
  answer_first    9/13 (0.692)


In [ ]:
# Per-transcript breakdown (matches the markdown summary the runner writes).
print(f"{'Source':45s} {'origin':>10s} {'FL':>9s} {'AF':>9s}")
print("-" * 78)
for ts in scored:
    fl_pass = sum(1 for r in ts.front_loading if r["passed"])
    fl_total = len(ts.front_loading)
    af_applicable = [r for r in ts.answer_first if r["applicable"]]
    af_pass = sum(1 for r in af_applicable if r["passed"])
    af_total = len(af_applicable)
    print(f"{ts.transcript.source_id:45s} {ts.transcript.origin:>10s} "
          f"{fl_pass:>3d}/{fl_total:<3d}  {af_pass:>3d}/{af_total:<3d}")


Source                                            origin        FL        AF
------------------------------------------------------------------------------
positive/01-novice-learns-prompting              authored    1/3      0/1  
positive/02-practitioner-evaluates-output        authored    2/3      0/0  
positive/03-ethical-dilemma-discussion           authored    4/7      0/0  
positive/04-appropriateness-judgment             authored    1/3      0/0  
negative/01-blind-trust-in-output                authored    2/7      2/2  
negative/02-over-reliance-full-delegation        authored    1/5      0/0  
negative/03-sharing-sensitive-data               authored    3/6      0/0  
simulated/fatigued-returner                     simulated    2/5      1/1  
simulated/novice-curious                        simulated    4/8      3/5  
simulated/skeptical-engineer                    simulated    3/6      3/4  


In [ ]:
# Pass rates by source label.
print(f"{'Label':12s} {'Front-Loading':>18s} {'Answer-First':>18s}")
print("-" * 50)
for label, m in sorted(aggregates["by_label"].items()):
    fl, af = m["front_loading"], m["answer_first"]
    fl_str = f"{fl['passed']}/{fl['applicable']} ({fl['rate']})"
    af_str = f"{af['passed']}/{af['applicable']} ({af['rate']})" if af['rate'] is not None else "n/a"
    print(f"{label:12s} {fl_str:>18s} {af_str:>18s}")

print()
print("Reading the results:")
print("- Front-Loading discriminates positive (0.5) > negative (0.333) — the rule-based")
print("  metric correctly flags negative authored transcripts as more often violating")
print("  the question/pre-pause discipline.")
print("- Answer-First on authored looks inverted (positive 0/1 vs negative 2/2), but")
print("  that's a small-sample artifact: only 1 positive applicable turn vs 2 negative.")
print("  The classifier filtered most learner messages because they were practice prompts,")
print("  not direct questions to SAGE.")
print("- Simulated sessions are the most useful test bed for Metric 2 (10 applicable")
print("  turns, 70% pass rate).")


Label        Front-Loading        Answer-First
--------------------------------------------------
negative      6/18 (0.333)          2/2 (1.0)
positive       8/16 (0.5)          0/1 (0.0)
simulated   9/19 (0.474)          7/10 (0.7)

Reading the results:
- Front-Loading discriminates positive (0.5) > negative (0.333) — the rule-based
  metric correctly flags negative authored transcripts as more often violating
  the question/pre-pause discipline.
- Answer-First on authored looks inverted (positive 0/1 vs negative 2/2), but
  that's a small-sample artifact: only 1 positive applicable turn vs 2 negative.
  The classifier filtered most learner messages because they were practice prompts,
  not direct questions to SAGE.
- Simulated sessions are the most useful test bed for Metric 2 (10 applicable
  turns, 70% pass rate).


## Step 2 — Extrinsic Evaluation (User Study)

- Recruit >=3 participants who are not in this class.
- Write a user study protocol including:
  - **Evaluation objective**: What aspect are you thinking of evaluating?
  - **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.
  - **Tasks**: What scenario(s) will they complete with your agent? (e.g., plan a weekend trip to a city with budget constraints)
  - **Metrics and analysis**: What will you track and how are you converting them into analysis (e.g., task success rate, time taken, satisfaction rating)?
  - **Baselines** (if any): What are you comparing against?
  - **Interview questions / pre or post survey design**: Short debrief to capture qualitative feedback (e.g., “What part of the interaction was most helpful?”, “What would you change?”)

Run the study and collect both quantitative and qualitative data. **To do so, make sure you use appropriate tools**. e.g. You may write code to collect user clickstreams/logs, use Google Form for survey questions, etc.



Useful links/references for running better studies:
- Competitive Research Methods, Chapter 5, by Elizabeth Goodman, Mike Kuniavsky and Andrea Moed in Observing the User Experience
- [Universal tools- Recruiting and Interviewing, Chapter 6](https://drive.google.com/file/d/1E7CgPEpqxVZmIR0YaaxWbt1NjqpgeICv/view?usp=drive_link) by Elizabeth Goodman, Mike Kuniavsky and Andrea Moed in Observing the User Experience
- [Example Interview Guide- Reading Ahead Interview Guide](https://drive.google.com/file/d/1jFwxgdco0oiaN7wFnSPxvJvL8jgCoeY9/view?usp=drive_link) by Portigal Consulting

### TODO: Please fill in study protocol here.

- **Evaluation objective**: Evaluate how the agent provides information; does it answer your question before asking another? Does the agent only ask one question at a time? Is the agent concise? The agent can ‘demo’ prompts against itself. After asking for prompt guidance, ask it to demo two different prompts.
- **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.
- **Tasks**: Make sure only one question is asked at a time. Make sure that the agent answers the question you provided before asking one of its own questions.
- **Metrics and analysis**: Rate your satisfaction score with how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.
- **Baselines** (if any): Optional: compare against any other AI out there, write thoughts and what agent you compared against.
- **Interview questions / pre or post survey design**: What would you change? (anything) Did this agent feel helpful? Was the agent invasive in question?


Please either copy and paste your chat session, or take screenshots of your chat session using print screen or the print feature on the top left dropdown.


In [ ]:
## Write code to analyze your data here.

# load data...

# see raw patterns...

# get visualization...

# Run stats...

## Conclusion...

# Just for fun, an example of a data analysis agent and some "agent recipies"
# i.e. patterns other people have created:
# https://huggingface.co/learn/cookbook/en/agent_data_analyst

## Step 3: Summarize & Reflect

Here, you will complete your writeup by reflecting on your evaluation, in these three parts:


### **Evaluation overview**

Create a SPHERE Evaluation Card for your evaluation. To do so,
- Carefully read [this paper](https://arxiv.org/abs/2504.07971) (you can ignore the appendix except for the examples starting on p.24)
- Go to [this website](https://sphere-eval.github.io/) and create the card by filling in the corresponding textboxes.
- Put the exported Json here:
  ```json
  {
    "sysname": "agent",
    "aspects": {
    ...
  ```

### **Experiment Results**

Summarize your learning from each evaluation, and propose one more evaluation you could think of running for this agent.

1. Intrinsic evaluation XXXX
  - Describe what was evaluated:
  - Present the key results (tables, figures, or example outputs):
  - Write at least one conclusion you can draw from the study:

2. Intrinsic evaluation XXXX
  - Describe what was evaluated:
  - Present the key results (tables, figures, or example outputs):
  - Write at least one conclusion you can draw from the study:

3. Exrinsic evaluation XXXX
  - Describe what was evaluated:
  - Present the key results (tables, figures, or example outputs):
  - Write at least one conclusion you can draw from the study:

## **Potential evaluation that could also be run**
- Briefly describe this evaluation:
- Explain why it’s relevant:
- Specify if it’s intrinsic or extrinsic:
- Outline the metric or study design:

### **Additional insights**

(Optional, but I would love to hear your insights and reflection on this process evaluating the agent.)

# Submission

1. Make sure all the important execution results are saved as files and then correctly displayed in whatever file you turn in, in a way that would allow me to grade you without re-running code.
2. Submit a link to your readme and/or evaluation write-up file in your repo (or a google doc)
3. Submit on brightspace a link to your write-up (in markdown in your project github repo, or in this notebook).

**Forgetting about this will result in deduction in the clarity and effort scores.**